# Reverse Validation: eICU-CRD Development → MIMIC-IV External Validation

## Sensitivity Analysis for Reviewer Comment #1

The original analysis trains on MIMIC-IV (single center, BIDMC) and validates on eICU-CRD (208 hospitals).
This notebook reverses the roles: **training on eICU-CRD** and **validating on MIMIC-IV**
to confirm that the observation-process feature effect on domain shift is not an artifact of training on a single idiosyncratic site.

All model specifications, preprocessing pipelines, and evaluation metrics are identical to `03_delivation_and_external_validation.ipynb`.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Imputation
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer

# Classification
from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    average_precision_score, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve

# Visualization
import matplotlib.pyplot as plt
import matplotlib

# Word document output
from docx import Document
from docx.shared import Pt
from docx.enum.table import WD_TABLE_ALIGNMENT

# Recalibration
from scipy.special import expit

# XGBoost
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

from pathlib import Path
from scipy.stats import spearmanr

# Warnings
warnings.filterwarnings('ignore')

# Settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully")

In [ ]:
# ============================================================
# Configurable Bootstrap Parameters
# ============================================================
# Set to 100 for development; change to 2000 for final manuscript
N_BOOT = 2000
print(f"Bootstrap resamples: N_BOOT = {N_BOOT}")

## Step 2: Load Pre-Processed Wide-Format Data

We load the same parquet files produced by `03_delivation_and_external_validation.ipynb`
to guarantee identical feature engineering.

In [ ]:
df_eicu = pd.read_parquet('../outputs/outputs_data/df_eicu_wide_with_count.parquet')
df_mimic = pd.read_parquet('../outputs/outputs_data/df_mimic_wide_with_count.parquet')

print(f"eICU-CRD (Development):          {df_eicu.shape[0]:,} patients, {df_eicu.shape[1]} columns")
print(f"  Mortality: {df_eicu['hospital_expire_flag'].mean()*100:.1f}%")
print(f"MIMIC-IV (External Validation):  {df_mimic.shape[0]:,} patients, {df_mimic.shape[1]} columns")
print(f"  Mortality: {df_mimic['hospital_expire_flag'].mean()*100:.1f}%")

## Step 3: Prepare X/y and Train/Test Split (REVERSED)

**Key difference from original**: eICU-CRD is now split 60/40 for development, and MIMIC-IV serves as the external validation set.

In [ ]:
TARGET = 'hospital_expire_flag'

# Drop non-feature columns (including subgroup variables - used for grouping only)
drop_cols = ['stay_id', 'sex', 'admit', 'vent', 'arf', TARGET,
             'race_ethnicity', 'teachingstatus', 'region', 'hospitalid']

# Development set: eICU-CRD (60/40 stratified split)
X_eicu = df_eicu.drop(columns=[c for c in drop_cols if c in df_eicu.columns])
y_eicu = df_eicu[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X_eicu, y_eicu,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y_eicu
)

# Save subgroup labels for potential subgroup analysis
subgroup_test_race = df_eicu.loc[X_test.index, 'race_ethnicity'].reset_index(drop=True)
subgroup_ext_race = df_mimic['race_ethnicity'].reset_index(drop=True)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# External validation set: MIMIC-IV (100%)
X_ext = df_mimic.drop(columns=[c for c in drop_cols if c in df_mimic.columns])
y_ext = df_mimic[TARGET]

print(f"Train (eICU 60%):                X={X_train.shape}, y={y_train.shape}, mortality={y_train.mean():.3f}")
print(f"Internal Test (eICU 40%):        X={X_test.shape}, y={y_test.shape}, mortality={y_test.mean():.3f}")
print(f"External Validation (MIMIC-IV):  X={X_ext.shape}, y={y_ext.shape}, mortality={y_ext.mean():.3f}")
print(f"\nMissing values:")
print(f"  Train:    {X_train.isnull().sum().sum():,}")
print(f"  Test:     {X_test.isnull().sum().sum():,}")
print(f"  External: {X_ext.isnull().sum().sum():,}")

## Step 4: Define Feature Sets

Identical feature definitions as the original analysis.

In [ ]:
# Model 1: APACHE III Only
MODEL1_NUMERICAL_FEATURES = ['apache3_score']
MODEL1_CATEGORICAL_FEATURES = []

# Model 2: Latest + Comorbidity
MODEL2_NUMERICAL_FEATURES = [
    'age', 'hr_latest', 'map_latest', 'temp_latest', 'rr_latest', 'gcs_latest', 'uop',
    'wbc_latest', 'hct_latest', 'sodium_latest', 'glucose_latest',
    'bun_latest', 'scr_latest', 'bili_latest', 'albumin_latest',
    'ph_latest', 'pao2_latest', 'pco2_latest', 'fio2_latest', 'aa_grad_latest'
]
MODEL2_CATEGORICAL_FEATURES = ['comorbidity']

# Count features (measurement frequency)
COUNT_FEATURES = [
    'hr_count', 'map_count', 'temp_count', 'rr_count', 'gcs_count',
    'wbc_count', 'hct_count', 'sodium_count', 'glucose_count',
    'bun_count', 'scr_count', 'bili_count', 'albumin_count',
    'ph_count', 'pao2_count', 'pco2_count', 'fio2_count', 'aa_grad_count'
]

# Model 3: Latest + Count + Comorbidity
MODEL3_NUMERICAL_FEATURES = MODEL2_NUMERICAL_FEATURES + COUNT_FEATURES
MODEL3_CATEGORICAL_FEATURES = ['comorbidity']

# Model 4: Min/Max + Comorbidity
MODEL4_NUMERICAL_FEATURES = [
    'age',
    'hr_min', 'hr_max', 'map_min', 'map_max', 'temp_min', 'temp_max',
    'rr_min', 'rr_max', 'gcs_min', 'gcs_max', 'uop',
    'wbc_min', 'wbc_max', 'hct_min', 'hct_max',
    'sodium_min', 'sodium_max', 'glucose_min', 'glucose_max',
    'bun_min', 'bun_max', 'scr_min', 'scr_max', 'bili_min', 'bili_max',
    'albumin_min', 'albumin_max',
    'ph_min', 'ph_max', 'pao2_min', 'pao2_max', 'pco2_min', 'pco2_max',
    'fio2_min', 'fio2_max', 'aa_grad_min', 'aa_grad_max'
]
MODEL4_CATEGORICAL_FEATURES = ['comorbidity']

# Model 5: Min/Max + Count + Comorbidity
MODEL5_NUMERICAL_FEATURES = MODEL4_NUMERICAL_FEATURES + COUNT_FEATURES
MODEL5_CATEGORICAL_FEATURES = ['comorbidity']

# Diff variables (calculated after imputation)
DIFF_VARS = ['hr', 'map', 'temp', 'rr', 'gcs',
             'wbc', 'hct', 'sodium', 'glucose', 'bun', 'scr', 'bili', 'albumin',
             'ph', 'pao2', 'pco2', 'fio2', 'aa_grad']

print("Feature definitions loaded")
print(f"  Model 1: {len(MODEL1_NUMERICAL_FEATURES)} numerical")
print(f"  Model 2: {len(MODEL2_NUMERICAL_FEATURES)} numerical + {len(MODEL2_CATEGORICAL_FEATURES)} categorical")
print(f"  Model 3: {len(MODEL3_NUMERICAL_FEATURES)} numerical + {len(MODEL3_CATEGORICAL_FEATURES)} categorical")
print(f"  Model 4: {len(MODEL4_NUMERICAL_FEATURES)} numerical + {len(MODEL4_CATEGORICAL_FEATURES)} categorical")
print(f"  Model 5: {len(MODEL5_NUMERICAL_FEATURES)} numerical + {len(MODEL5_CATEGORICAL_FEATURES)} categorical")

## Step 5: Create Pipeline Functions

In [ ]:
def create_pipeline(numerical_features, categorical_features, random_state=42):
    """Create pipeline with IterativeImputer for numerical and SimpleImputer for categorical."""
    numerical_pipeline = Pipeline([
        ('imputer', IterativeImputer(max_iter=10, random_state=random_state)),
        ('scaler', StandardScaler())
    ])
    transformers = [('num', numerical_pipeline, numerical_features)]
    if categorical_features:
        categorical_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_pipeline, categorical_features))
    preprocessor = ColumnTransformer(transformers)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=random_state))
    ])


def create_pipeline_no_impute(numerical_features, categorical_features, random_state=42):
    """Create pipeline without imputation (for pre-imputed diff features)."""
    numerical_pipeline = Pipeline([('scaler', StandardScaler())])
    transformers = [('num', numerical_pipeline, numerical_features)]
    if categorical_features:
        categorical_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_pipeline, categorical_features))
    preprocessor = ColumnTransformer(transformers)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=random_state))
    ])


def create_xgb_pipeline(numerical_features, categorical_features, random_state=42):
    """Create XGBoost pipeline with SimpleImputer (for Models 1-5)."""
    numerical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    transformers = [('num', numerical_transformer, numerical_features)]
    if categorical_features:
        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, categorical_features))
    preprocessor = ColumnTransformer(transformers)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(random_state=random_state, eval_metric='logloss', n_jobs=-1))
    ])

def create_xgb_pipeline_no_impute(numerical_features, categorical_features, random_state=42):
    """Create XGBoost pipeline WITHOUT imputation (for Models 6-7 with pre-imputed diff features)."""
    transformers = [('num', StandardScaler(), numerical_features)]
    if categorical_features:
        categorical_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformers.append(('cat', categorical_transformer, categorical_features))
    preprocessor = ColumnTransformer(transformers)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(random_state=random_state, eval_metric='logloss', n_jobs=-1))
    ])

print("Pipeline functions defined")

## Step 6: Prepare Model-Specific Data

In [ ]:
# Prepare data for each model (train, test, external)
X_train_m1 = X_train[MODEL1_NUMERICAL_FEATURES + MODEL1_CATEGORICAL_FEATURES].copy()
X_test_m1 = X_test[MODEL1_NUMERICAL_FEATURES + MODEL1_CATEGORICAL_FEATURES].copy()
X_ext_m1 = X_ext[MODEL1_NUMERICAL_FEATURES + MODEL1_CATEGORICAL_FEATURES].copy()

X_train_m2 = X_train[MODEL2_NUMERICAL_FEATURES + MODEL2_CATEGORICAL_FEATURES].copy()
X_test_m2 = X_test[MODEL2_NUMERICAL_FEATURES + MODEL2_CATEGORICAL_FEATURES].copy()
X_ext_m2 = X_ext[MODEL2_NUMERICAL_FEATURES + MODEL2_CATEGORICAL_FEATURES].copy()

X_train_m3 = X_train[MODEL3_NUMERICAL_FEATURES + MODEL3_CATEGORICAL_FEATURES].copy()
X_test_m3 = X_test[MODEL3_NUMERICAL_FEATURES + MODEL3_CATEGORICAL_FEATURES].copy()
X_ext_m3 = X_ext[MODEL3_NUMERICAL_FEATURES + MODEL3_CATEGORICAL_FEATURES].copy()

X_train_m4 = X_train[MODEL4_NUMERICAL_FEATURES + MODEL4_CATEGORICAL_FEATURES].copy()
X_test_m4 = X_test[MODEL4_NUMERICAL_FEATURES + MODEL4_CATEGORICAL_FEATURES].copy()
X_ext_m4 = X_ext[MODEL4_NUMERICAL_FEATURES + MODEL4_CATEGORICAL_FEATURES].copy()

X_train_m5 = X_train[MODEL5_NUMERICAL_FEATURES + MODEL5_CATEGORICAL_FEATURES].copy()
X_test_m5 = X_test[MODEL5_NUMERICAL_FEATURES + MODEL5_CATEGORICAL_FEATURES].copy()
X_ext_m5 = X_ext[MODEL5_NUMERICAL_FEATURES + MODEL5_CATEGORICAL_FEATURES].copy()

print("Model-specific datasets prepared")
for i, (name, X) in enumerate([
    ('Model 1', X_train_m1), ('Model 2', X_train_m2), ('Model 3', X_train_m3),
    ('Model 4', X_train_m4), ('Model 5', X_train_m5)
], 1):
    print(f"  {name}: {X.shape[1]} features")

## Step 7: Train Logistic Regression Models 1–5

In [ ]:
print("Training Logistic Regression Models 1-5 on eICU-CRD...")

# Model 1
print("\nModel 1 (APACHE III Only)...")
pipeline_m1 = create_pipeline(MODEL1_NUMERICAL_FEATURES, MODEL1_CATEGORICAL_FEATURES, RANDOM_STATE)
pipeline_m1.fit(X_train_m1, y_train)
print("  Done")

# Model 2
print("\nModel 2 (Latest + Comorbidity)...")
pipeline_m2 = create_pipeline(MODEL2_NUMERICAL_FEATURES, MODEL2_CATEGORICAL_FEATURES, RANDOM_STATE)
pipeline_m2.fit(X_train_m2, y_train)

pipeline_m3 = create_pipeline(MODEL3_NUMERICAL_FEATURES, MODEL3_CATEGORICAL_FEATURES, RANDOM_STATE)
pipeline_m3.fit(X_train_m3, y_train)
print("  Model 3 (Latest + Count + Comorbidity): Done")
print("  Done")

# Model 4
print("\nModel 4 (Min/Max + Comorbidity)...")
pipeline_m4 = create_pipeline(MODEL4_NUMERICAL_FEATURES, MODEL4_CATEGORICAL_FEATURES, RANDOM_STATE)
pipeline_m4.fit(X_train_m4, y_train)
print("  Done")

# Model 5
print("\nModel 5 (Min/Max + Count + Comorbidity)...")
pipeline_m5 = create_pipeline(MODEL5_NUMERICAL_FEATURES, MODEL5_CATEGORICAL_FEATURES, RANDOM_STATE)
pipeline_m5.fit(X_train_m5, y_train)
print("  Done")

## Step 8: Create Diff Features and Train Models 6–7

Models 6-7 use pre-imputed difference features (variability = max − min).

In [ ]:
# Pre-impute numerical features for diff calculation
imputer_for_diff = IterativeImputer(max_iter=10, random_state=RANDOM_STATE)

X_train_num = X_train[MODEL4_NUMERICAL_FEATURES].copy()
X_test_num = X_test[MODEL4_NUMERICAL_FEATURES].copy()
X_ext_num = X_ext[MODEL4_NUMERICAL_FEATURES].copy()

X_train_imputed = pd.DataFrame(
    imputer_for_diff.fit_transform(X_train_num),
    columns=MODEL4_NUMERICAL_FEATURES,
    index=X_train.index
)
X_test_imputed = pd.DataFrame(
    imputer_for_diff.transform(X_test_num),
    columns=MODEL4_NUMERICAL_FEATURES,
    index=X_test.index
)
X_ext_imputed = pd.DataFrame(
    imputer_for_diff.transform(X_ext_num),
    columns=MODEL4_NUMERICAL_FEATURES,
    index=X_ext.index
)

def create_diff_features(df_imputed, df_original):
    """Create diff (variability) features from imputed min/max values."""
    result = pd.DataFrame(index=df_imputed.index)
    result['age'] = df_imputed['age']
    result['uop'] = df_imputed['uop']
    for var in DIFF_VARS:
        max_col = f'{var}_max'
        min_col = f'{var}_min'
        if max_col in df_imputed.columns and min_col in df_imputed.columns:
            result[f'{var}_diff'] = df_imputed[max_col] - df_imputed[min_col]
    for var in DIFF_VARS:
        count_col = f'{var}_count'
        if count_col in df_original.columns:
            result[count_col] = df_original[count_col].values
    if 'comorbidity' in df_original.columns:
        result['comorbidity'] = df_original['comorbidity'].values
    return result

X_train_diff = create_diff_features(X_train_imputed, X_train)
X_test_diff = create_diff_features(X_test_imputed, X_test)
X_ext_diff = create_diff_features(X_ext_imputed, X_ext)

# Define Model 6 & 7 feature lists
MODEL6_NUM_FEATURES = ['age', 'uop'] + [f'{var}_diff' for var in DIFF_VARS if f'{var}_diff' in X_train_diff.columns]
MODEL6_CAT_FEATURES = ['comorbidity']

MODEL7_NUM_FEATURES = MODEL6_NUM_FEATURES + [f'{var}_count' for var in DIFF_VARS if f'{var}_count' in X_train_diff.columns]
MODEL7_CAT_FEATURES = ['comorbidity']

X_train_m6 = X_train_diff[MODEL6_NUM_FEATURES + MODEL6_CAT_FEATURES].copy()
X_test_m6 = X_test_diff[MODEL6_NUM_FEATURES + MODEL6_CAT_FEATURES].copy()
X_ext_m6 = X_ext_diff[MODEL6_NUM_FEATURES + MODEL6_CAT_FEATURES].copy()

X_train_m7 = X_train_diff[MODEL7_NUM_FEATURES + MODEL7_CAT_FEATURES].copy()
X_test_m7 = X_test_diff[MODEL7_NUM_FEATURES + MODEL7_CAT_FEATURES].copy()
X_ext_m7 = X_ext_diff[MODEL7_NUM_FEATURES + MODEL7_CAT_FEATURES].copy()

print(f"  Model 6: {len(MODEL6_NUM_FEATURES)} numerical + {len(MODEL6_CAT_FEATURES)} categorical")
print(f"  Model 7: {len(MODEL7_NUM_FEATURES)} numerical + {len(MODEL7_CAT_FEATURES)} categorical")

In [ ]:
# Model 6
print("\nModel 6 (Diff + Comorbidity)...")
pipeline_m6 = create_pipeline_no_impute(MODEL6_NUM_FEATURES, MODEL6_CAT_FEATURES, RANDOM_STATE)
pipeline_m6.fit(X_train_m6, y_train)
print("  Done")

# Model 7
print("\nModel 7 (Diff + Count + Comorbidity)...")
pipeline_m7 = create_pipeline_no_impute(MODEL7_NUM_FEATURES, MODEL7_CAT_FEATURES, RANDOM_STATE)
pipeline_m7.fit(X_train_m7, y_train)
print("  Done")

## Step 9: Model Registry

In [ ]:
# Store all pipelines
pipelines = {
    'Model 1': {'name': 'APACHE III Only', 'pipeline': pipeline_m1,
                'X_train': X_train_m1, 'X_test': X_test_m1, 'X_ext': X_ext_m1, 'y_test': y_test, 'y_ext': y_ext},
    'Model 2': {'name': 'Latest + Comorbidity', 'pipeline': pipeline_m2,
                'X_train': X_train_m2, 'X_test': X_test_m2, 'X_ext': X_ext_m2, 'y_test': y_test, 'y_ext': y_ext},
    'Model 3': {'name': 'Latest + Count + Comorbidity', 'pipeline': pipeline_m3,
                'X_train': X_train_m3, 'X_test': X_test_m3, 'X_ext': X_ext_m3, 'y_test': y_test, 'y_ext': y_ext},
    'Model 4': {'name': 'Min/Max + Comorbidity', 'pipeline': pipeline_m4,
                'X_train': X_train_m4, 'X_test': X_test_m4, 'X_ext': X_ext_m4, 'y_test': y_test, 'y_ext': y_ext},
    'Model 5': {'name': 'Min/Max + Count + Comorbidity', 'pipeline': pipeline_m5,
                'X_train': X_train_m5, 'X_test': X_test_m5, 'X_ext': X_ext_m5, 'y_test': y_test, 'y_ext': y_ext},
    'Model 6': {'name': 'Diff + Comorbidity', 'pipeline': pipeline_m6,
                'X_train': X_train_m6, 'X_test': X_test_m6, 'X_ext': X_ext_m6, 'y_test': y_test, 'y_ext': y_ext},
    'Model 7': {'name': 'Diff + Count + Comorbidity', 'pipeline': pipeline_m7,
                'X_train': X_train_m7, 'X_test': X_test_m7, 'X_ext': X_ext_m7, 'y_test': y_test, 'y_ext': y_ext},
}

print("Model registry created with 7 logistic regression models")

## Step 10: XGBoost Models with Hyperparameter Tuning

In [ ]:
# --- Hyperparameter grid ---
xgb_param_grid = {
    'classifier__n_estimators': [100, 200, 300, 400, 500],
    'classifier__max_depth': [3, 4, 5, 6, 7, 8, 9, 10],
    'classifier__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
    'classifier__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'classifier__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'classifier__min_child_weight': [1, 3, 5, 7, 10],
    'classifier__gamma': [0, 0.1, 0.5, 1, 5],
    'classifier__reg_alpha': [0, 0.01, 0.1, 0.5, 1],
    'classifier__reg_lambda': [0, 0.1, 1, 5, 10],
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Model configurations ---
model_configs = {
    'Model 1': {'num': MODEL1_NUMERICAL_FEATURES, 'cat': MODEL1_CATEGORICAL_FEATURES,
                'X_train': pipelines['Model 1']['X_train'], 'use_impute': True},
    'Model 2': {'num': MODEL2_NUMERICAL_FEATURES, 'cat': MODEL2_CATEGORICAL_FEATURES,
                'X_train': pipelines['Model 2']['X_train'], 'use_impute': True},
    'Model 3': {'num': MODEL3_NUMERICAL_FEATURES, 'cat': MODEL3_CATEGORICAL_FEATURES,
                'X_train': pipelines['Model 3']['X_train'], 'use_impute': True},
    'Model 4': {'num': MODEL4_NUMERICAL_FEATURES, 'cat': MODEL4_CATEGORICAL_FEATURES,
                'X_train': pipelines['Model 4']['X_train'], 'use_impute': True},
    'Model 5': {'num': MODEL5_NUMERICAL_FEATURES, 'cat': MODEL5_CATEGORICAL_FEATURES,
                'X_train': pipelines['Model 5']['X_train'], 'use_impute': True},
    'Model 6': {'num': MODEL6_NUM_FEATURES, 'cat': MODEL6_CAT_FEATURES,
                'X_train': pipelines['Model 6']['X_train'], 'use_impute': False},
    'Model 7': {'num': MODEL7_NUM_FEATURES, 'cat': MODEL7_CAT_FEATURES,
                'X_train': pipelines['Model 7']['X_train'], 'use_impute': False},
}

# --- Train all XGBoost models ---
print('=' * 60)
print('XGBOOST MODEL TRAINING WITH HYPERPARAMETER TUNING')
print('=' * 60)

xgb_pipelines = {}

for model_key, config in model_configs.items():
    desc = pipelines[model_key]['name']
    print(f'\nTuning XGB {model_key}: {desc}...')
    
    if config['use_impute']:
        xgb_pipe = create_xgb_pipeline(config['num'], config['cat'], RANDOM_STATE)
    else:
        xgb_pipe = create_xgb_pipeline_no_impute(config['num'], config['cat'], RANDOM_STATE)
    
    search = RandomizedSearchCV(
        estimator=xgb_pipe,
        param_distributions=xgb_param_grid,
        n_iter=30,
        scoring='roc_auc',
        cv=cv_strategy,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )
    search.fit(config['X_train'], y_train)
    xgb_pipelines[model_key] = search.best_estimator_
    print(f'  Best CV AUC: {search.best_score_:.4f}')

print('\n' + '=' * 60)
print('All XGBoost models trained')
print('=' * 60)

## Step 11: Bootstrap and Calibration Functions

In [ ]:
def calculate_calibration_metrics(y_true, y_pred):
    """Calculate calibration slope and intercept (CITL) via logistic recalibration:
    logit(Y) = alpha + beta * logit(p_hat).  Ideal: CITL=0, slope=1.
    """
    y_pred_clipped = np.clip(y_pred, 1e-10, 1 - 1e-10)
    logit_pred = np.log(y_pred_clipped / (1 - y_pred_clipped))
    lr_cal = LogisticRegression(solver='lbfgs', max_iter=1000)
    lr_cal.fit(logit_pred.reshape(-1, 1), y_true)
    return lr_cal.coef_[0][0], lr_cal.intercept_[0]


def bootstrap_ci(y_true, y_pred, metric_func, n_bootstrap=None, ci=0.95, random_state=42):
    """Calculate bootstrap confidence interval for a metric."""
    if n_bootstrap is None:
        n_bootstrap = N_BOOT
    rng = np.random.RandomState(random_state)
    n = len(y_true)
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    point_estimate = metric_func(y_true_arr, y_pred_arr)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yt = y_true_arr[idx]
        yp = y_pred_arr[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            scores.append(metric_func(yt, yp))
        except Exception:
            continue
    alpha = 1 - ci
    lo = np.percentile(scores, alpha / 2 * 100)
    hi = np.percentile(scores, (1 - alpha / 2) * 100)
    return point_estimate, lo, hi


def bootstrap_delta_ci(y_true_int, y_pred_int, y_true_ext, y_pred_ext,
                       metric_func, n_bootstrap=None, ci=0.95, random_state=42):
    """Bootstrap CI for delta = metric(external) - metric(internal).
    Resamples independently within each dataset.
    """
    if n_bootstrap is None:
        n_bootstrap = N_BOOT
    rng = np.random.RandomState(random_state)
    yti = np.asarray(y_true_int)
    ypi = np.asarray(y_pred_int)
    yte = np.asarray(y_true_ext)
    ype = np.asarray(y_pred_ext)
    n_int = len(yti)
    n_ext = len(yte)
    point_int = metric_func(yti, ypi)
    point_ext = metric_func(yte, ype)
    point_delta = point_ext - point_int
    deltas = []
    for _ in range(n_bootstrap):
        ii = rng.randint(0, n_int, n_int)
        ie = rng.randint(0, n_ext, n_ext)
        yt_i = yti[ii]
        yp_i = ypi[ii]
        yt_e = yte[ie]
        yp_e = ype[ie]
        if len(np.unique(yt_i)) < 2 or len(np.unique(yt_e)) < 2:
            continue
        try:
            deltas.append(metric_func(yt_e, yp_e) - metric_func(yt_i, yp_i))
        except Exception:
            continue
    alpha = 1 - ci
    lo = np.percentile(deltas, alpha / 2 * 100)
    hi = np.percentile(deltas, (1 - alpha / 2) * 100)
    return point_delta, lo, hi


def bootstrap_calibration_ci(y_true, y_pred, n_bootstrap=None, ci=0.95, random_state=42):
    """Bootstrap CI for calibration slope and CITL (intercept).
    Returns: (slope, slope_lo, slope_hi, intercept, int_lo, int_hi)
    """
    if n_bootstrap is None:
        n_bootstrap = N_BOOT
    rng = np.random.RandomState(random_state)
    n = len(y_true)
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    point_slope, point_int = calculate_calibration_metrics(yt, yp)
    slopes = []
    intercepts = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        yti = yt[idx]
        ypi = yp[idx]
        if len(np.unique(yti)) < 2:
            continue
        try:
            s, i = calculate_calibration_metrics(yti, ypi)
            slopes.append(s)
            intercepts.append(i)
        except Exception:
            continue
    alpha = 1 - ci
    lo_p = alpha / 2 * 100
    hi_p = (1 - alpha / 2) * 100
    return (point_slope, np.percentile(slopes, lo_p), np.percentile(slopes, hi_p),
            point_int, np.percentile(intercepts, lo_p), np.percentile(intercepts, hi_p))

print(f'Bootstrap iterations: N_BOOT = {N_BOOT}')

## Step 12: Publication-Quality Outputs — Table 2 (Reverse)

In [ ]:
# ============================================================
# Publication Outputs: Reverse Domain Shift Performance Table
# ============================================================
OUT_DIR = Path('../outputs/outputs_for_reverse_validation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = ['Model 1', 'Model 2', 'Model 3', 'Model 4', 'Model 5', 'Model 6', 'Model 7']

# --- Helper: format values with CI ---
def fmt_ci(point, lo, hi):
    return f"{point:.3f} ({lo:.3f}\u2013{hi:.3f})"

def fmt_delta_ci(point, lo, hi):
    sign = '+' if point >= 0 else ''
    return f"{sign}{point:.3f} ({lo:.3f}\u2013{hi:.3f})"

# --- Helper: save DataFrame as Word table ---
def save_table_as_docx(df, title, footnotes_str, filepath):
    doc = Document()
    # Title
    p = doc.add_paragraph()
    run = p.add_run(title)
    run.bold = True
    run.font.size = Pt(11)
    # Table
    table = doc.add_table(rows=len(df) + 1, cols=len(df.columns), style='Table Grid')
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, col in enumerate(df.columns):
        cell = table.rows[0].cells[j]
        cell.text = str(col)
        for paragraph in cell.paragraphs:
            for run in paragraph.runs:
                run.bold = True
                run.font.size = Pt(8)
    for i, (_, row) in enumerate(df.iterrows()):
        for j, val in enumerate(row):
            cell = table.rows[i + 1].cells[j]
            cell.text = str(val)
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(8)
    doc.add_paragraph()
    p = doc.add_paragraph()
    run = p.add_run(footnotes_str)
    run.font.size = Pt(8)
    run.italic = True
    doc.save(filepath)

# --- Compute Table 2 (Reverse) ---
print("=" * 60)
print("Computing Table 2 (Reverse): Domain Shift Performance")
print("=" * 60)

rows_table2 = []
for algo_name in ['Logistic regression', 'XGBoost']:
    for mk in MODEL_ORDER:
        config = pipelines[mk]
        print(f"  {algo_name} - {mk}...")

        if algo_name == 'Logistic regression':
            pipe = config['pipeline']
        else:
            if mk not in xgb_pipelines:
                print(f"    Skipped (no XGBoost model)")
                continue
            pipe = xgb_pipelines[mk]

        y_pred_train = pipe.predict_proba(config['X_train'])[:, 1]
        y_pred_test = pipe.predict_proba(config['X_test'])[:, 1]
        y_pred_ext = pipe.predict_proba(config['X_ext'])[:, 1]
        n_predictors = len(config['X_train'].columns)

        # Train AUC (for overfitting gap)
        auc_train = roc_auc_score(y_train, y_pred_train)

        # Internal test metrics with CI
        auc_test, auc_test_lo, auc_test_hi = bootstrap_ci(y_test, y_pred_test, roc_auc_score)
        ap_test, ap_test_lo, ap_test_hi = bootstrap_ci(y_test, y_pred_test, average_precision_score)
        brier_test, brier_test_lo, brier_test_hi = bootstrap_ci(y_test, y_pred_test, brier_score_loss)

        # External metrics with CI
        auc_ext, auc_ext_lo, auc_ext_hi = bootstrap_ci(y_ext, y_pred_ext, roc_auc_score)
        ap_ext, ap_ext_lo, ap_ext_hi = bootstrap_ci(y_ext, y_pred_ext, average_precision_score)
        brier_ext, brier_ext_lo, brier_ext_hi = bootstrap_ci(y_ext, y_pred_ext, brier_score_loss)

        # Delta metrics with CI
        d_auc, d_auc_lo, d_auc_hi = bootstrap_delta_ci(y_test, y_pred_test, y_ext, y_pred_ext, roc_auc_score)
        d_ap, d_ap_lo, d_ap_hi = bootstrap_delta_ci(y_test, y_pred_test, y_ext, y_pred_ext, average_precision_score)
        d_brier, d_brier_lo, d_brier_hi = bootstrap_delta_ci(y_test, y_pred_test, y_ext, y_pred_ext, brier_score_loss)

        # Calibration with CI
        slope, slope_lo, slope_hi, citl, citl_lo, citl_hi = bootstrap_calibration_ci(y_ext, y_pred_ext)

        rows_table2.append({
            'Algorithm': algo_name,
            'Model ID': mk,
            'Feature set (specification)': config['name'],
            '# Predictors': n_predictors,
            'AUROC (Internal, 95% CI)': fmt_ci(auc_test, auc_test_lo, auc_test_hi),
            'AUROC (External, 95% CI)': fmt_ci(auc_ext, auc_ext_lo, auc_ext_hi),
            '\u0394AUROC (External \u2212 Internal, 95% CI)': fmt_delta_ci(d_auc, d_auc_lo, d_auc_hi),
            'AUPRC (Internal, 95% CI)': fmt_ci(ap_test, ap_test_lo, ap_test_hi),
            'AUPRC (External, 95% CI)': fmt_ci(ap_ext, ap_ext_lo, ap_ext_hi),
            '\u0394AUPRC (External \u2212 Internal, 95% CI)': fmt_delta_ci(d_ap, d_ap_lo, d_ap_hi),
            'Brier (Internal, 95% CI)': fmt_ci(brier_test, brier_test_lo, brier_test_hi),
            'Brier (External, 95% CI)': fmt_ci(brier_ext, brier_ext_lo, brier_ext_hi),
            '\u0394Brier (External \u2212 Internal, 95% CI)': fmt_delta_ci(d_brier, d_brier_lo, d_brier_hi),
            'Calibration intercept in external (CITL, 95% CI)': fmt_ci(citl, citl_lo, citl_hi),
            'Calibration slope in external (95% CI)': fmt_ci(slope, slope_lo, slope_hi),
            'Overfitting gap (Train \u2212 Internal AUROC)': round(auc_train - auc_test, 3),
        })

df_table2 = pd.DataFrame(rows_table2)
display(df_table2)

# Footnotes (updated for reverse direction)
prev_int = y_test.mean()
prev_ext = y_ext.mean()
FOOTNOTES_TABLE2 = (
    f"Abbreviations: AUROC, area under the receiver operating characteristic curve; "
    f"AUPRC, area under the precision\u2013recall curve; CITL, calibration-in-the-large; CI, confidence interval.\n"
    f"1. Internal validation: held-out eICU-CRD test set (40%). External validation: independent MIMIC-IV cohort.\n"
    f"2. \u0394 = External \u2212 Internal. Negative \u0394 indicates performance degradation (domain shift).\n"
    f"3. 95% CIs: bootstrap resampling (B = {N_BOOT}, percentile method). \u0394 CIs from bootstrap distribution of the difference.\n"
    f"4. Calibration via logistic recalibration: logit(Y) = \u03b1 + \u03b2\u00b7logit(p\u0302). "
    f"Ideal: CITL = 0, slope = 1.\n"
    f"5. Brier score: lower is better. \u0394Brier > 0 = worse external performance.\n"
    f"6. Outcome prevalence: internal = {prev_int:.1%}; external = {prev_ext:.1%}. "
    f"AUPRC is prevalence-dependent.\n"
    f"7. Overfitting gap = Train AUROC \u2212 Internal AUROC.\n"
    f"8. Consistent preprocessing across internal and external datasets."
)

# Save .md
md_text = "### S.Table. Reverse validation: eICU-CRD development to MIMIC-IV external validation\n\n"
md_text += df_table2.to_markdown(index=False)
md_text += "\n\n" + FOOTNOTES_TABLE2
with open(OUT_DIR / 'Table_domain_shift_performance_reverse.md', 'w', encoding='utf-8') as f:
    f.write(md_text)
print("\nSaved: Table_domain_shift_performance_reverse.md")

# Save .docx
save_table_as_docx(
    df_table2,
    'S.Table. Reverse validation: eICU-CRD development to MIMIC-IV external validation',
    FOOTNOTES_TABLE2,
    OUT_DIR / 'Table_domain_shift_performance_reverse.docx'
)
print("Saved: Table_domain_shift_performance_reverse.docx")

## Step 13: Publication Figures

In [ ]:
# ============================================================
# Publication Figures (PNG only, 300 DPI)
# ============================================================
matplotlib.rcParams.update({
    'font.size': 10,
    'font.family': 'sans-serif',
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

# 2x4 layout: (row, col, model_key)
GRID_LAYOUT = [
    (0, 0, 'Model 1'), (0, 1, 'Model 2'), (0, 2, 'Model 4'), (0, 3, 'Model 6'),
    (1, 1, 'Model 3'), (1, 2, 'Model 5'), (1, 3, 'Model 7'),
]

def create_2x4_figure(figsize=(20, 10)):
    fig, axes = plt.subplots(2, 4, figsize=figsize)
    axes[1, 0].set_visible(False)
    return fig, axes

# ============================================================
# Figure: Delta AUROC vs Model (Reverse)
# ============================================================
print("Creating Figure: Delta AUROC (Reverse)...")
fig, ax = plt.subplots(figsize=(8, 5))
dodge = 0.15
for ai, (algo_label, get_pipe, color, marker) in enumerate([
    ('Logistic Regression', lambda mk: pipelines[mk]['pipeline'], '#4472C4', 'o'),
    ('XGBoost', lambda mk: xgb_pipelines.get(mk), '#ED7D31', 's'),
]):
    offset = -dodge / 2 + ai * dodge
    x_positions, y_vals, y_err_lo, y_err_hi = [], [], [], []
    for idx, mk in enumerate(MODEL_ORDER):
        config = pipelines[mk]
        pipe = get_pipe(mk)
        if pipe is None:
            continue
        yp_test = pipe.predict_proba(config['X_test'])[:, 1]
        yp_ext = pipe.predict_proba(config['X_ext'])[:, 1]
        d, dlo, dhi = bootstrap_delta_ci(y_test, yp_test, y_ext, yp_ext, roc_auc_score)
        x_positions.append(idx + offset)
        y_vals.append(d)
        y_err_lo.append(d - dlo)
        y_err_hi.append(dhi - d)
    ax.errorbar(x_positions, y_vals, yerr=[y_err_lo, y_err_hi], fmt='', marker=marker, linestyle='none',
                color=color, label=algo_label, capsize=4, linewidth=1.5, markersize=6)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
ax.set_xlabel('Model')
ax.set_ylabel('\u0394AUROC (External \u2212 Internal)')
ax.set_title('Performance Drop (\u0394AUROC) vs Model (Reverse: eICU \u2192 MIMIC)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(OUT_DIR / 'Figure_delta_AUROC_vs_complexity_reverse.png')
print("Saved: Figure_delta_AUROC_vs_complexity_reverse.png")
plt.show()

# ============================================================
# Figure: Delta AUPRC vs Model (Reverse)
# ============================================================
print("Creating Figure: Delta AUPRC (Reverse)...")
fig, ax = plt.subplots(figsize=(8, 5))
dodge = 0.15
for ai, (algo_label, get_pipe, color, marker) in enumerate([
    ('Logistic Regression', lambda mk: pipelines[mk]['pipeline'], '#4472C4', 'o'),
    ('XGBoost', lambda mk: xgb_pipelines.get(mk), '#ED7D31', 's'),
]):
    offset = -dodge / 2 + ai * dodge
    x_positions, y_vals, y_err_lo, y_err_hi = [], [], [], []
    for idx, mk in enumerate(MODEL_ORDER):
        config = pipelines[mk]
        pipe = get_pipe(mk)
        if pipe is None:
            continue
        yp_test = pipe.predict_proba(config['X_test'])[:, 1]
        yp_ext = pipe.predict_proba(config['X_ext'])[:, 1]
        d, dlo, dhi = bootstrap_delta_ci(y_test, yp_test, y_ext, yp_ext, average_precision_score)
        x_positions.append(idx + offset)
        y_vals.append(d)
        y_err_lo.append(d - dlo)
        y_err_hi.append(dhi - d)
    ax.errorbar(x_positions, y_vals, yerr=[y_err_lo, y_err_hi], fmt='', marker=marker, linestyle='none',
                color=color, label=algo_label, capsize=4, linewidth=1.5, markersize=6)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER, rotation=45, ha="right")
ax.set_xlabel('Model')
ax.set_ylabel('\u0394AUPRC (External \u2212 Internal)')
ax.set_title('Performance Drop (\u0394AUPRC) vs Model (Reverse: eICU \u2192 MIMIC)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(OUT_DIR / 'Figure_delta_AUPRC_vs_complexity_reverse.png')
print("Saved: Figure_delta_AUPRC_vs_complexity_reverse.png")
plt.show()

# ============================================================
# Calibration Figures (LR and XGBoost)
# ============================================================
for algo_label, get_pipe, fname in [
    ('Logistic Regression', lambda mk: pipelines[mk]['pipeline'], 'Figure_Calibration_external_LR_reverse.png'),
    ('XGBoost', lambda mk: xgb_pipelines.get(mk), 'Figure_Calibration_external_XGB_reverse.png'),
]:
    print(f"Creating Calibration Figure ({algo_label})...")
    fig, axes = create_2x4_figure(figsize=(20, 10))
    for r, c, mk in GRID_LAYOUT:
        ax = axes[r, c]
        config = pipelines[mk]
        pipe = get_pipe(mk)
        if pipe is None:
            ax.set_visible(False)
            continue
        ax2 = ax.twinx()
        ax2.set_ylabel('')
        ax2.set_yticks([])
        ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1.0,
                label='Perfect', zorder=3)
        for label, X, y_true, color, marker_s in [
            ('Derivation', config['X_train'], y_train, '#2CA02C', 'o'),
            ('Internal',   config['X_test'],  y_test,  '#1F77B4', '^'),
            ('External',   config['X_ext'],   y_ext,   '#D62728', 's'),
        ]:
            y_pred = pipe.predict_proba(X)[:, 1]
            ax2.hist(y_pred, bins=30, range=(0, 1), alpha=0.25, color=color, zorder=0)
            frac_pos, mean_pred = calibration_curve(y_true, y_pred, n_bins=10, strategy='uniform')
            brier = brier_score_loss(y_true, y_pred)
            slope_val, _ = calculate_calibration_metrics(y_true, y_pred)
            ax.plot(mean_pred, frac_pos, marker=marker_s, linestyle='-', color=color,
                    linewidth=1.5, markersize=5, markeredgecolor=color, markerfacecolor=color,
                    label=f'{label} (Brier={brier:.3f}, slope={slope_val:.2f})')
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.set_title(f'{mk}: {config["name"]}')
        ax.legend(loc='lower right', fontsize=7)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        for spine in ax.spines.values():
            spine.set_linewidth(0.6)
    fig.suptitle(f'Calibration \u2014 Derivation / Internal / External ({algo_label}, Reverse: eICU \u2192 MIMIC)',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig.savefig(OUT_DIR / fname)
    print(f"Saved: {fname}")
    plt.show()

# ============================================================
# ROC Figures (LR and XGBoost)
# ============================================================
for algo_label, get_pipe, fname in [
    ('Logistic Regression', lambda mk: pipelines[mk]['pipeline'], 'SFigure_ROC_test_vs_external_LR_reverse.png'),
    ('XGBoost', lambda mk: xgb_pipelines.get(mk), 'SFigure_ROC_test_vs_external_XGB_reverse.png'),
]:
    print(f"Creating ROC Figure ({algo_label})...")
    fig, axes = create_2x4_figure()
    for r, c, mk in GRID_LAYOUT:
        ax = axes[r, c]
        config = pipelines[mk]
        pipe = get_pipe(mk)
        if pipe is None:
            ax.set_visible(False)
            continue
        yp_test = pipe.predict_proba(config['X_test'])[:, 1]
        fpr_t, tpr_t, _ = roc_curve(y_test, yp_test)
        auc_t = roc_auc_score(y_test, yp_test)
        yp_ext = pipe.predict_proba(config['X_ext'])[:, 1]
        fpr_e, tpr_e, _ = roc_curve(y_ext, yp_ext)
        auc_e = roc_auc_score(y_ext, yp_ext)
        ax.plot(fpr_t, tpr_t, '-', color='#4472C4', label=f'Internal (AUC={auc_t:.3f})', linewidth=1.5)
        ax.plot(fpr_e, tpr_e, '--', color='#ED7D31', label=f'External (AUC={auc_e:.3f})', linewidth=1.5)
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=0.8)
        ax.set_xlabel('1 \u2212 Specificity')
        ax.set_ylabel('Sensitivity')
        ax.set_title(f'{mk}: {config["name"]}')
        ax.legend(loc='lower right', fontsize=8)
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
    fig.suptitle(f'ROC Curves \u2014 Internal Test vs External ({algo_label}, Reverse: eICU \u2192 MIMIC)', fontsize=13, y=1.01)
    plt.tight_layout()
    fig.savefig(OUT_DIR / fname)
    print(f"Saved: {fname}")
    plt.show()

# ============================================================
# PRC Figures (LR and XGBoost)
# ============================================================
for algo_label, get_pipe, fname in [
    ('Logistic Regression', lambda mk: pipelines[mk]['pipeline'], 'SFigure_PRC_external_LR_reverse.png'),
    ('XGBoost', lambda mk: xgb_pipelines.get(mk), 'SFigure_PRC_external_XGB_reverse.png'),
]:
    print(f"Creating PRC Figure ({algo_label})...")
    fig, axes = create_2x4_figure()
    for r, c, mk in GRID_LAYOUT:
        ax = axes[r, c]
        config = pipelines[mk]
        pipe = get_pipe(mk)
        if pipe is None:
            ax.set_visible(False)
            continue
        for label, X, y_true, ls, clr in [
            ('Internal', config['X_test'], y_test, '-', '#4472C4'),
            ('External', config['X_ext'], y_ext, '--', '#ED7D31'),
        ]:
            yp = pipe.predict_proba(X)[:, 1]
            prec, rec, _ = precision_recall_curve(y_true, yp)
            ap = average_precision_score(y_true, yp)
            ax.plot(rec, prec, ls, color=clr, label=f'{label} (AP={ap:.3f})', linewidth=1.5)
        prev_int_val = y_test.mean()
        prev_ext_val = y_ext.mean()
        ax.axhline(y=prev_int_val, color='#4472C4', linestyle=':', alpha=0.4, linewidth=0.8)
        ax.axhline(y=prev_ext_val, color='#ED7D31', linestyle=':', alpha=0.4, linewidth=0.8)
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title(f'{mk}: {config["name"]}')
        ax.legend(loc='upper right', fontsize=8)
        ax.set_xlim([-0.02, 1.02])
        ax.set_ylim([-0.02, 1.02])
    fig.suptitle(f'Precision\u2013Recall Curves \u2014 Internal vs External ({algo_label}, Reverse: eICU \u2192 MIMIC)', fontsize=13, y=1.01)
    plt.tight_layout()
    fig.savefig(OUT_DIR / fname)
    print(f"Saved: {fname}")
    plt.show()

print("\nAll publication figures saved.")

## Step 14: Recalibration Table (Reverse)

In [ ]:
# ============================================================
# S.Table: Recalibration Results (Reverse)
# ============================================================
print("=" * 60)
print("Creating S.Table: Recalibration Results (Reverse)")
print("=" * 60)

s2_rows = []
for algo_name in ['Logistic regression', 'XGBoost']:
    for mk in MODEL_ORDER:
        config = pipelines[mk]
        print(f"  {algo_name} - {mk}...")

        if algo_name == 'Logistic regression':
            pipe = config['pipeline']
        else:
            if mk not in xgb_pipelines:
                continue
            pipe = xgb_pipelines[mk]

        y_pred_ext = pipe.predict_proba(config['X_ext'])[:, 1]

        # Pre-recalibration
        slope_pre, citl_pre = calculate_calibration_metrics(y_ext, y_pred_ext)
        brier_pre = brier_score_loss(y_ext, y_pred_ext)
        logloss_pre = log_loss(y_ext, y_pred_ext)

        # Fit recalibration model: logit(Y) = alpha + beta * logit(p_hat)
        y_pred_clipped = np.clip(y_pred_ext, 1e-10, 1 - 1e-10)
        logit_pred = np.log(y_pred_clipped / (1 - y_pred_clipped))
        lr_recal = LogisticRegression(solver='lbfgs', max_iter=1000)
        lr_recal.fit(logit_pred.reshape(-1, 1), y_ext)
        alpha_recal = lr_recal.intercept_[0]
        beta_recal = lr_recal.coef_[0][0]
        y_pred_recal = expit(alpha_recal + beta_recal * logit_pred)

        # Post-recalibration metrics
        brier_post = brier_score_loss(y_ext, y_pred_recal)
        logloss_post = log_loss(y_ext, y_pred_recal)

        s2_rows.append({
            'Algorithm': algo_name,
            'Model ID': mk,
            'Feature set': config['name'],
            'CITL (pre-recal)': f"{citl_pre:.3f}",
            'Slope (pre-recal)': f"{slope_pre:.3f}",
            'Recalibrated \u03b1': f"{alpha_recal:.3f}",
            'Recalibrated \u03b2': f"{beta_recal:.3f}",
            'Brier (pre)': f"{brier_pre:.4f}",
            'Brier (post)': f"{brier_post:.4f}",
            '\u0394Brier': f"{brier_post - brier_pre:.4f}",
            'Log loss (pre)': f"{logloss_pre:.4f}",
            'Log loss (post)': f"{logloss_post:.4f}",
            '\u0394Log loss': f"{logloss_post - logloss_pre:.4f}",
        })

df_stable2 = pd.DataFrame(s2_rows)
display(df_stable2)

# Footnotes
stable2_footnotes = (
    "Abbreviations: CITL, calibration-in-the-large; LR, logistic regression; XGB, XGBoost.\n"
    "Recalibration model: logit(Y) = \u03b1 + \u03b2 \u00b7 logit(p\u0302). "
    "Ideal pre-recalibration: \u03b1 = 0, \u03b2 = 1. "
    "Negative \u0394Brier / \u0394Log loss indicates improvement after recalibration."
)

# Save .md
md_text = "### S.Table. Recalibration-only results \u2014 Reverse validation (external: MIMIC-IV)\n\n"
md_text += df_stable2.to_markdown(index=False)
md_text += "\n\n" + stable2_footnotes
with open(OUT_DIR / 'STable_recalibration_reverse.md', 'w', encoding='utf-8') as f:
    f.write(md_text)
print("\nSaved: STable_recalibration_reverse.md")

# Save .docx
save_table_as_docx(df_stable2,
    'S.Table. Recalibration-only results \u2014 Reverse validation (external: MIMIC-IV)',
    stable2_footnotes, OUT_DIR / 'STable_recalibration_reverse.docx')
print("Saved: STable_recalibration_reverse.docx")

## Step 15: Comparison with Forward Results

This is the key deliverable for the reviewer: comparing domain shift patterns in both directions.

In [ ]:
# ============================================================
# Load Forward Results from Table_domain_shift_performance.md
# ============================================================
import re

forward_path = Path('../outputs/outputs_for_manuscript/Table_domain_shift_performance.md')
with open(forward_path, 'r', encoding='utf-8') as f:
    forward_text = f.read()

# Parse markdown table
lines = [l.strip() for l in forward_text.strip().split('\n') if l.strip().startswith('|')]
header = [h.strip() for h in lines[0].split('|')[1:-1]]
# Skip separator line (lines[1])
forward_rows = []
for line in lines[2:]:
    vals = [v.strip() for v in line.split('|')[1:-1]]
    forward_rows.append(dict(zip(header, vals)))
df_forward = pd.DataFrame(forward_rows)

print(f"Loaded forward results: {len(df_forward)} rows")
display(df_forward[['Algorithm', 'Model ID', '\u0394AUROC (External \u2212 Internal, 95% CI)', '\u0394AUPRC (External \u2212 Internal, 95% CI)']].head(14))

In [ ]:
# ============================================================
# Build Comparison Table
# ============================================================

def parse_delta_ci(text):
    """Parse 'sign0.XXX (0.XXX\u20130.XXX)' into (point, lo, hi)."""
    text = text.replace('\u2013', '-').replace('\u2212', '-')
    m = re.match(r'([+\-]?[0-9.]+)\s*\(([+\-]?[0-9.]+)[-\u2013]([+\-]?[0-9.]+)\)', text)
    if m:
        return float(m.group(1)), float(m.group(2)), float(m.group(3))
    return None, None, None

comparison_rows = []
for _, fwd_row in df_forward.iterrows():
    algo = fwd_row['Algorithm']
    mk = fwd_row['Model ID']
    feature_set = fwd_row['Feature set (specification)']

    # Forward delta
    fwd_auroc_text = fwd_row.get('\u0394AUROC (External \u2212 Internal, 95% CI)', '')
    fwd_auprc_text = fwd_row.get('\u0394AUPRC (External \u2212 Internal, 95% CI)', '')
    fwd_auroc, fwd_auroc_lo, fwd_auroc_hi = parse_delta_ci(fwd_auroc_text)
    fwd_auprc, fwd_auprc_lo, fwd_auprc_hi = parse_delta_ci(fwd_auprc_text)

    # Reverse delta (from df_table2)
    rev_match = df_table2[(df_table2['Algorithm'] == algo) & (df_table2['Model ID'] == mk)]
    if len(rev_match) == 0:
        continue
    rev_row = rev_match.iloc[0]
    rev_auroc_text = rev_row['\u0394AUROC (External \u2212 Internal, 95% CI)']
    rev_auprc_text = rev_row['\u0394AUPRC (External \u2212 Internal, 95% CI)']
    rev_auroc, rev_auroc_lo, rev_auroc_hi = parse_delta_ci(rev_auroc_text)
    rev_auprc, rev_auprc_lo, rev_auprc_hi = parse_delta_ci(rev_auprc_text)

    comparison_rows.append({
        'Algorithm': algo,
        'Model ID': mk,
        'Feature set': feature_set,
        'Forward \u0394AUROC (MIMIC\u2192eICU)': fwd_auroc_text,
        'Reverse \u0394AUROC (eICU\u2192MIMIC)': rev_auroc_text,
        'Forward \u0394AUPRC (MIMIC\u2192eICU)': fwd_auprc_text,
        'Reverse \u0394AUPRC (eICU\u2192MIMIC)': rev_auprc_text,
        '_fwd_auroc': fwd_auroc,
        '_fwd_auroc_lo': fwd_auroc_lo,
        '_fwd_auroc_hi': fwd_auroc_hi,
        '_rev_auroc': rev_auroc,
        '_rev_auroc_lo': rev_auroc_lo,
        '_rev_auroc_hi': rev_auroc_hi,
    })

df_comparison = pd.DataFrame(comparison_rows)

# Display comparison table (publication columns only)
pub_cols = ['Algorithm', 'Model ID', 'Feature set',
            'Forward \u0394AUROC (MIMIC\u2192eICU)', 'Reverse \u0394AUROC (eICU\u2192MIMIC)',
            'Forward \u0394AUPRC (MIMIC\u2192eICU)', 'Reverse \u0394AUPRC (eICU\u2192MIMIC)']
display(df_comparison[pub_cols])

# Spearman correlation
valid = df_comparison.dropna(subset=['_fwd_auroc', '_rev_auroc'])
rho, p_val = spearmanr(valid['_fwd_auroc'], valid['_rev_auroc'])
print(f"\nSpearman correlation between Forward and Reverse \u0394AUROC: r = {rho:.3f} (p = {p_val:.4f})")

# Save comparison table
df_pub = df_comparison[pub_cols]

# Save .md
comp_title = "### S.Table. Comparison of forward and reverse validation \u0394AUROC and \u0394AUPRC\n\n"
comp_footnotes = (
    f"Forward: MIMIC-IV development \u2192 eICU-CRD external validation. "
    f"Reverse: eICU-CRD development \u2192 MIMIC-IV external validation.\n"
    f"\u0394 = External \u2212 Internal. Negative values indicate performance degradation.\n"
    f"Spearman correlation between forward and reverse \u0394AUROC: r = {rho:.3f} (p = {p_val:.4f}).\n"
    f"95% CIs: bootstrap resampling (B = {N_BOOT}, percentile method)."
)
md_text = comp_title + df_pub.to_markdown(index=False) + "\n\n" + comp_footnotes
with open(OUT_DIR / 'Table_comparison_forward_vs_reverse.md', 'w', encoding='utf-8') as f:
    f.write(md_text)
print("Saved: Table_comparison_forward_vs_reverse.md")

# Save .docx
save_table_as_docx(df_pub,
    'S.Table. Comparison of forward and reverse validation \u0394AUROC and \u0394AUPRC',
    comp_footnotes, OUT_DIR / 'Table_comparison_forward_vs_reverse.docx')
print("Saved: Table_comparison_forward_vs_reverse.docx")

In [ ]:
# ============================================================
# Forest Plot: Forward vs Reverse Delta AUROC
# ============================================================
print("Creating comparison forest plot...")

fig, ax = plt.subplots(figsize=(10, 7))

# Separate by algorithm
y_pos = 0
y_ticks = []
y_labels = []
spacing = 1.0
algo_gap = 0.5

for algo_name, algo_color_fwd, algo_color_rev in [
    ('Logistic regression', '#4472C4', '#7BA3D9'),
    ('XGBoost', '#ED7D31', '#F4B183'),
]:
    algo_df = df_comparison[df_comparison['Algorithm'] == algo_name].reset_index(drop=True)
    if y_pos > 0:
        y_pos += algo_gap
    for _, row in algo_df.iterrows():
        mk = row['Model ID']

        # Forward
        fwd_pt = row['_fwd_auroc']
        fwd_lo = row['_fwd_auroc_lo']
        fwd_hi = row['_fwd_auroc_hi']
        if fwd_pt is not None:
            ax.errorbar(fwd_pt, y_pos + 0.12, xerr=[[fwd_pt - fwd_lo], [fwd_hi - fwd_pt]],
                        fmt='o', color=algo_color_fwd, markersize=7, capsize=3, linewidth=1.5,
                        label='Forward (MIMIC\u2192eICU)' if y_pos == 0 else '')

        # Reverse
        rev_pt = row['_rev_auroc']
        rev_lo = row['_rev_auroc_lo']
        rev_hi = row['_rev_auroc_hi']
        if rev_pt is not None:
            ax.errorbar(rev_pt, y_pos - 0.12, xerr=[[rev_pt - rev_lo], [rev_hi - rev_pt]],
                        fmt='^', color=algo_color_rev, markersize=7, capsize=3, linewidth=1.5,
                        label='Reverse (eICU\u2192MIMIC)' if y_pos == 0 else '')

        y_ticks.append(y_pos)
        y_labels.append(f'{algo_name[:3]}. {mk}')
        y_pos += spacing

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=1.0)
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels)
ax.invert_yaxis()
ax.set_xlabel('\u0394AUROC (External \u2212 Internal)')
ax.set_title('Forward vs Reverse Validation: \u0394AUROC Comparison\n'
             f'Spearman r = {rho:.3f} (p = {p_val:.4f})')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='#4472C4', linestyle='none', markersize=7, label='Forward LR (MIMIC\u2192eICU)'),
    Line2D([0], [0], marker='^', color='#7BA3D9', linestyle='none', markersize=7, label='Reverse LR (eICU\u2192MIMIC)'),
    Line2D([0], [0], marker='o', color='#ED7D31', linestyle='none', markersize=7, label='Forward XGB (MIMIC\u2192eICU)'),
    Line2D([0], [0], marker='^', color='#F4B183', linestyle='none', markersize=7, label='Reverse XGB (eICU\u2192MIMIC)'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9)

plt.tight_layout()
fig.savefig(OUT_DIR / 'Figure_comparison_delta_AUROC_both_directions.png', dpi=300, bbox_inches='tight')
print("Saved: Figure_comparison_delta_AUROC_both_directions.png")
plt.show()

## Step 16: Summary Statistics for Manuscript

In [ ]:
# ============================================================
# Key comparisons for manuscript text
# ============================================================
print("=" * 60)
print("SUMMARY: Key Comparisons for Manuscript")
print("=" * 60)

print("\n--- Logistic Regression ---")
for mk in MODEL_ORDER:
    fwd = df_comparison[(df_comparison['Algorithm'] == 'Logistic regression') & (df_comparison['Model ID'] == mk)]
    if len(fwd) > 0:
        row = fwd.iloc[0]
        print(f"  {mk} ({row['Feature set']}):")
        print(f"    Forward \u0394AUROC = {row['_fwd_auroc']:.3f}, Reverse \u0394AUROC = {row['_rev_auroc']:.3f}")

print("\n--- XGBoost ---")
for mk in MODEL_ORDER:
    fwd = df_comparison[(df_comparison['Algorithm'] == 'XGBoost') & (df_comparison['Model ID'] == mk)]
    if len(fwd) > 0:
        row = fwd.iloc[0]
        print(f"  {mk} ({row['Feature set']}):")
        print(f"    Forward \u0394AUROC = {row['_fwd_auroc']:.3f}, Reverse \u0394AUROC = {row['_rev_auroc']:.3f}")

print(f"\nSpearman correlation: r = {rho:.3f}, p = {p_val:.4f}")

# Check consistency: do models with counts consistently show larger degradation?
print("\n--- Pattern Check: Count features \u2192 larger degradation? ---")
paired_models = [
    ('Model 2', 'Model 3', 'Latest \u00b1 Count'),
    ('Model 4', 'Model 5', 'Min/Max \u00b1 Count'),
    ('Model 6', 'Model 7', 'Diff \u00b1 Count'),
]
for m_no_count, m_with_count, desc in paired_models:
    for algo in ['Logistic regression', 'XGBoost']:
        no_c = df_comparison[(df_comparison['Algorithm'] == algo) & (df_comparison['Model ID'] == m_no_count)]
        with_c = df_comparison[(df_comparison['Algorithm'] == algo) & (df_comparison['Model ID'] == m_with_count)]
        if len(no_c) > 0 and len(with_c) > 0:
            d_no = no_c.iloc[0]['_rev_auroc']
            d_with = with_c.iloc[0]['_rev_auroc']
            larger = d_with < d_no  # more negative = larger degradation
            print(f"  {algo[:3]}. {desc}: without={d_no:.3f}, with={d_with:.3f}, "
                  f"count worsens: {'YES' if larger else 'NO'}")

print("\n" + "=" * 60)
print("OUTPUT SUMMARY")
print("=" * 60)
for f in sorted(OUT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<55} {size:>8.1f} KB")